In [1]:
import os

os.environ['KAGGLE_USERNAME'] = 'hariprasanthu5002'
os.environ['KAGGLE_KEY'] = 'KGAT_87ed6356b64ede123c082703ad93a6b8'



In [ ]:
kaggle datasets list -s plantvillage

HTTPSConnectionPool(host='www.kaggle.com', port=443): Max retries exceeded with url: /api/v1/datasets/list?group=public&sortby=hottest&size=all&filetype=all&license=all&viewed=unspecified&search=plantvillage&page=1 (Caused by NameResolutionError("<urllib3.connection.HTTPSConnection object at 0x00000257F09E59C0>: Failed to resolve 'www.kaggle.com' ([Errno 11001] getaddrinfo failed)"))


In [3]:
!kaggle datasets download -d abdallahalidev/plantvillage-dataset



HTTPSConnectionPool(host='www.kaggle.com', port=443): Max retries exceeded with url: /api/v1/datasets/metadata/abdallahalidev/plantvillage-dataset (Caused by NameResolutionError("<urllib3.connection.HTTPSConnection object at 0x000001651BFD5B70>: Failed to resolve 'www.kaggle.com' ([Errno 11001] getaddrinfo failed)"))


In [4]:
!unzip -q plantvillage-dataset.zip


'unzip' is not recognized as an internal or external command,
operable program or batch file.


In [5]:
import shutil

shutil.rmtree("/content/plantvillage dataset/grayscale")
shutil.rmtree("/content/plantvillage dataset/segmented")

print("Unused folders removed")


FileNotFoundError: [WinError 3] The system cannot find the path specified: '/content/plantvillage dataset/grayscale'

In [ ]:
import os

dataset_dir = "/content/plantvillage dataset/color"

classes = [d for d in os.listdir(dataset_dir)
           if os.path.isdir(os.path.join(dataset_dir, d))]

print("Number of classes:", len(classes))


Number of classes: 38


In [ ]:
import os
from sklearn.model_selection import train_test_split

dataset_dir = "/content/plantvillage dataset/color"

# Collect image paths and labels
image_paths = []
labels = []

for cls in os.listdir(dataset_dir):
    cls_path = os.path.join(dataset_dir, cls)
    if os.path.isdir(cls_path):
        for file in os.listdir(cls_path):
            if file.lower().endswith(('.jpg', '.jpeg', '.png')):
                image_paths.append(os.path.join(cls_path, file))
                labels.append(cls)

                                                            # First split: train + temp
X_train, X_temp, y_train, y_temp = train_test_split(
      image_paths, labels, test_size=0.30, stratify=labels, random_state=42
)

X_val, X_test, y_val, y_test = train_test_split(
      X_temp, y_temp, test_size=0.50, stratify=y_temp, random_state=42
)

print("Train size:", len(X_train))
print("Validation size:", len(X_val))
print("Test size:", len(X_test))






                                                                                                                                #

Train size: 38013
Validation size: 8146
Test size: 8146


In [ ]:
import torchvision.transforms as transforms

train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

val_test_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])


In [ ]:
from torch.utils.data import Dataset
from PIL import Image

class PlantDataset(Dataset):
    def __init__(self, image_paths, labels, transform=None):
        self.image_paths = image_paths
        self.labels = labels
        self.transform = transform

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        img_path = self.image_paths[idx]
        label = self.labels[idx]

        image = Image.open(img_path).convert("RGB")
        label = class_to_idx[label]

        if self.transform:
            image = self.transform(image)

        return image, label


In [ ]:
train_dataset = PlantDataset(X_train, y_train, transform=train_transform)
val_dataset = PlantDataset(X_val, y_val, transform=val_test_transform)
test_dataset = PlantDataset(X_test, y_test, transform=val_test_transform)


In [ ]:
from torch.utils.data import DataLoader

batch_size = 32

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=2)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=2)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, num_workers=2)

print("DataLoaders ready.")


DataLoaders ready.


In [ ]:
class_names = sorted(list(set(labels)))
class_to_idx = {cls: idx for idx, cls in enumerate(class_names)}



In [ ]:
import torch
import torchvision.models as models
import torch.nn as nn

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

model = models.vit_b_16(weights="IMAGENET1K_V1")

num_classes = len(class_to_idx)

model.heads.head = nn.Linear(
    model.heads.head.in_features,
    num_classes
)

model = model.to(device)

print(model.heads.head)


Using device: cpu
Downloading: "https://download.pytorch.org/models/vit_b_16-c867db91.pth" to /root/.cache/torch/hub/checkpoints/vit_b_16-c867db91.pth


100%|██████████| 330M/330M [00:02<00:00, 160MB/s]


Linear(in_features=768, out_features=38, bias=True)


In [ ]:
images, labels_batch = next(iter(train_loader))
print("Label min:", labels_batch.min().item())
print("Label max:", labels_batch.max().item())


Label min: 1
Label max: 37


In [ ]:
import torch.optim as optim

criterion = nn.CrossEntropyLoss()
optimizer = optim.AdamW(model.parameters(), lr=3e-5, weight_decay=1e-4)


In [ ]:
import torch
from tqdm import tqdm
import matplotlib.pyplot as plt

# ================= TRAINING CONFIG =================
num_epochs = 20
patience = 5
best_val_acc = 0
epochs_no_improve = 0

train_losses = []
val_losses = []
train_accuracies = []
val_accuracies = []

for epoch in range(num_epochs):

    print(f"\nEpoch {epoch+1}/{num_epochs}")
    print("-" * 50)

    # ================= TRAINING =================
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0

    for images, labels in tqdm(train_loader):

        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        outputs = model(images)
        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()

        running_loss += loss.item()

        _, predicted = torch.max(outputs, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

    train_loss = running_loss / len(train_loader)
    train_acc = 100 * correct / total

    train_losses.append(train_loss)
    train_accuracies.append(train_acc)

    # ================= VALIDATION =================
    model.eval()
    val_running_loss = 0.0
    val_correct = 0
    val_total = 0

    with torch.no_grad():
        for images, labels in tqdm(val_loader):

            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)
            loss = criterion(outputs, labels)

            val_running_loss += loss.item()

            _, predicted = torch.max(outputs, 1)
            val_total += labels.size(0)
            val_correct += (predicted == labels).sum().item()

    val_loss = val_running_loss / len(val_loader)
    val_acc = 100 * val_correct / val_total

    val_losses.append(val_loss)
    val_accuracies.append(val_acc)

    print(f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.2f}%")
    print(f"Val   Loss: {val_loss:.4f} | Val   Acc: {val_acc:.2f}%")

    # ================= SAVE BEST MODEL =================
    if val_acc > best_val_acc:
        print("Validation improved. Saving best model.")
        best_val_acc = val_acc
        torch.save(model.state_dict(), "best_model.pth")
        epochs_no_improve = 0
    else:
        epochs_no_improve += 1
        print(f"No improvement for {epochs_no_improve} epoch(s)")

    # ================= EARLY STOPPING =================
    if epochs_no_improve >= patience:
        print("Early stopping triggered.")
        break

print("\nTraining Complete.")
print(f"Best Validation Accuracy: {best_val_acc:.2f}%")

# ================= LOAD BEST MODEL =================
model.load_state_dict(torch.load("best_model.pth"))
model = model.to(device)



Epoch 1/20
--------------------------------------------------


  0%|          | 2/1188 [02:20<23:05:10, 70.08s/it]